# CP8305 Final Project
## Notebook 5 – Unsupervised Modelling (Diabetes 130-US Hospitals)

### Objectives
- Visualise the high-dimensional patient space with PCA, t-SNE, and (optionally) UMAP.
- Discover patient phenotype clusters via K-Means, Agglomerative Clustering, and Gaussian Mixture Models.
- Validate cluster count with elbow, silhouette, BIC/AIC, and dendrogram analyses.
- Profile each cluster by clinical feature means and readmission rates.
- Mine readmission-focused association rules (FP-Growth) to surface interpretable clinical patterns.
- Bridge unsupervised structure with the supervised readmission target from Notebook 3.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score

sys.path.insert(0, str(Path('..').resolve()))
TABLES_DIR = Path('..') / 'reports' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from src.modeling import find_optimal_k, train_kmeans
from src.visualization import (
    plot_elbow_curve,
    plot_cluster_scatter,
    plot_association_rules,
    save_figure,
)

np.random.seed(42)
sns.set_theme(style='whitegrid')
%matplotlib inline

## 2. Load Processed Data

In [ ]:
DATA_FILE = Path('..') / 'data' / 'diabetes130' / 'processed' / 'processed_dataset.csv'
df = pd.read_csv(DATA_FILE)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
TARGET = 'readmitted_binary'

drop_columns = [TARGET]
for leakage_col in ['encounter_id', 'patient_nbr']:
    if leakage_col in df.columns:
        drop_columns.append(leakage_col)

X = df.drop(columns=drop_columns)
y = df[TARGET]

print(f'Features: {X.shape[1]}  |  Target: {y.name}  |  Positive rate: {y.mean():.3f}')

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
print(f'Scaled matrix: {X_scaled.shape}')

## 3. Dimensionality Reduction and Visualization

### 3.1 PCA – Explained Variance

In [ ]:
N_COMPONENTS_FULL = min(X_scaled.shape)
pca_full = PCA(n_components=N_COMPONENTS_FULL, random_state=42)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

for threshold in (0.80, 0.90, 0.95):
    n = int(np.searchsorted(cumvar, threshold) + 1)
    print(f'{threshold:.0%} variance explained by {n} / {X_scaled.shape[1]} components')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, color='steelblue', linewidth=1.2)
for thr, ls in [(0.80, ':'), (0.90, '--'), (0.95, '-.')]:
    ax.axhline(thr, color='grey', linestyle=ls, linewidth=0.8, label=f'{thr:.0%}')
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA – Cumulative Explained Variance')
ax.legend(loc='lower right')
ax.set_xlim(1, min(300, len(cumvar)))
fig.tight_layout()
save_figure(fig, '05_pca_variance_explained.png')
plt.show()

### 3.2 PCA – 2-D Projection

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 7))
for label, colour, marker in [(0, 'steelblue', 'o'), (1, 'crimson', 'x')]:
    mask = y == label
    ax.scatter(
        X_pca2[mask, 0], X_pca2[mask, 1],
        c=colour, marker=marker, alpha=0.25, s=8,
        label=f'{"Readmitted" if label else "Not readmitted"}',
    )

n_arrows = 8
loadings = pca2.components_.T
loading_norms = np.linalg.norm(loadings, axis=1)
top_idx = loading_norms.argsort()[-n_arrows:]
scale = np.abs(X_pca2).max() * 0.6
for i in top_idx:
    ax.annotate(
        X.columns[i],
        xy=(0, 0),
        xytext=(loadings[i, 0] * scale, loadings[i, 1] * scale),
        arrowprops=dict(arrowstyle='->', color='black', lw=0.8),
        fontsize=7, ha='center', va='center',
    )

ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA 2-D Projection coloured by Readmission Status')
ax.legend(loc='upper right', markerscale=3)
fig.tight_layout()
save_figure(fig, '05_pca_2d_projection.png')
plt.show()

### 3.3 t-SNE Embedding

PCA(50) is applied first to reduce compute cost, then t-SNE projects to 2-D.

In [ ]:
PCA_PRE = 50
TSNE_SAMPLE = min(10_000, len(X_scaled))

rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_scaled), size=TSNE_SAMPLE, replace=False)
X_sample = X_scaled.iloc[sample_idx]
y_sample = y.iloc[sample_idx]

pca_pre = PCA(n_components=PCA_PRE, random_state=42)
X_pca50 = pca_pre.fit_transform(X_sample)

tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_pca50)

fig, ax = plt.subplots(figsize=(9, 7))
for label, colour, marker in [(0, 'steelblue', 'o'), (1, 'crimson', 'x')]:
    mask = y_sample.values == label
    ax.scatter(
        X_tsne[mask, 0], X_tsne[mask, 1],
        c=colour, marker=marker, alpha=0.35, s=8,
        label=f'{"Readmitted" if label else "Not readmitted"}',
    )
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title(f't-SNE 2-D Embedding (n={TSNE_SAMPLE:,}, PCA-{PCA_PRE} pre-reduction)')
ax.legend(loc='upper right', markerscale=3)
fig.tight_layout()
save_figure(fig, '05_tsne_2d.png')
plt.show()

### 3.4 UMAP Embedding (Optional)

In [ ]:
try:
    import umap

    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    X_umap = reducer.fit_transform(X_pca50)

    fig, ax = plt.subplots(figsize=(9, 7))
    for label, colour, marker in [(0, 'steelblue', 'o'), (1, 'crimson', 'x')]:
        mask = y_sample.values == label
        ax.scatter(
            X_umap[mask, 0], X_umap[mask, 1],
            c=colour, marker=marker, alpha=0.35, s=8,
            label=f'{"Readmitted" if label else "Not readmitted"}',
        )
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(f'UMAP 2-D Embedding (n={TSNE_SAMPLE:,}, PCA-{PCA_PRE} pre-reduction)')
    ax.legend(loc='upper right', markerscale=3)
    fig.tight_layout()
    save_figure(fig, '05_umap_2d.png')
    plt.show()
except ImportError:
    print('umap-learn not installed — skipping UMAP visualisation.')

## 4. K-Means Clustering

The processed dataset has ~2,400 sparse one-hot columns. Clustering directly in this space produces degenerate results (near-uniform inertia, single-cluster convergence). We therefore cluster on **PCA-50** reduced data, which is standard practice for high-dimensional sparse features and consistent with the t-SNE and hierarchical sections.

### 4.1 Optimal *k* Selection (Elbow + Silhouette)

In [ ]:
CLUSTER_PCA = 50
pca_clust = PCA(n_components=CLUSTER_PCA, random_state=42)
X_clust = pd.DataFrame(
    pca_clust.fit_transform(X_scaled),
    index=X_scaled.index,
    columns=[f'PC{i+1}' for i in range(CLUSTER_PCA)],
)
print(f'Clustering on PCA-{CLUSTER_PCA} representation: {X_clust.shape}')

K_RANGE = range(2, 11)

elbow_data = find_optimal_k(X_clust, k_range=K_RANGE)
fig = plot_elbow_curve(elbow_data, title='K-Means Elbow Curve (PCA-50)')
save_figure(fig, '05_elbow_curve.png')
plt.show()

In [ ]:
MIN_CLUSTER_FRAC = 0.01

sil_scores = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X_clust)
    n_unique = len(np.unique(labels))
    if n_unique < 2:
        print(f'  k={k}  silhouette=N/A (only {n_unique} cluster found)')
        continue
    smallest_frac = np.bincount(labels).min() / len(labels)
    if smallest_frac < MIN_CLUSTER_FRAC:
        print(f'  k={k}  silhouette=skipped (smallest cluster is {smallest_frac:.4f} of data)')
        continue
    try:
        sil = silhouette_score(X_clust, labels)
        sil_scores.append((k, sil))
        print(f'  k={k}  silhouette={sil:.4f}')
    except ValueError as e:
        print(f'  k={k}  silhouette=N/A ({e})')

ks, sils = zip(*sil_scores)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, sils, marker='o', color='steelblue')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs. k (PCA-50)')
fig.tight_layout()
save_figure(fig, '05_silhouette_scores.png')
plt.show()

OPTIMAL_K = int(ks[np.argmax(sils)])
print(f'\nBest silhouette at k={OPTIMAL_K}')

### 4.2 Fit K-Means and Visualise on PCA Projection

In [ ]:
kmeans_model, cluster_labels = train_kmeans(X_clust, n_clusters=OPTIMAL_K)
df['cluster'] = cluster_labels

fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(
    X_pca2[:, 0], X_pca2[:, 1],
    c=cluster_labels, cmap='tab10', alpha=0.3, s=6,
)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'K-Means (k={OPTIMAL_K}) on PCA 2-D Projection')
legend = ax.legend(
    *scatter.legend_elements(),
    title='Cluster', loc='upper right',
)
fig.tight_layout()
save_figure(fig, '05_cluster_scatter_pca.png')
plt.show()

### 4.3 Cluster Profiling

In [ ]:
PROFILE_COLS = [
    c for c in [
        'time_in_hospital', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_outpatient', 'number_emergency',
        'number_inpatient', 'number_diagnoses',
    ]
    if c in df.columns
]

profile = df.groupby('cluster')[PROFILE_COLS].mean()
print('Cluster means (unscaled features):')
display(profile.round(2))

profile.to_csv(TABLES_DIR / '05_cluster_profiles.csv')
print(f'Saved to {TABLES_DIR / "05_cluster_profiles.csv"}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=False)
axes = axes.ravel()
for i, col in enumerate(PROFILE_COLS):
    ax = axes[i]
    df.boxplot(column=col, by='cluster', ax=ax, grid=False)
    ax.set_title(col)
    ax.set_xlabel('Cluster')
for j in range(len(PROFILE_COLS), len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Feature Distributions by K-Means Cluster', fontsize=14)
fig.tight_layout()
save_figure(fig, '05_cluster_profiles.png')
plt.show()

In [ ]:
ct = pd.crosstab(df['cluster'], y, margins=True)
print('Cluster × Readmission Crosstab:')
display(ct)

readmit_rate = df.groupby('cluster')[TARGET].mean()
chi2, p_val, dof, expected = stats.chi2_contingency(
    pd.crosstab(df['cluster'], y)
)
print(f'\nChi-squared = {chi2:.2f}, p = {p_val:.2e}, dof = {dof}')

fig, ax = plt.subplots(figsize=(7, 4))
readmit_rate.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.axhline(y.mean(), color='crimson', linestyle='--', label=f'Overall rate ({y.mean():.3f})')
ax.set_xlabel('Cluster')
ax.set_ylabel('30-Day Readmission Rate')
ax.set_title('Readmission Rate by K-Means Cluster')
ax.legend()
for i, v in enumerate(readmit_rate):
    ax.text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=9)
fig.tight_layout()
save_figure(fig, '05_cluster_readmission_rates.png')
plt.show()

## 5. Hierarchical (Agglomerative) Clustering

Ward-linkage agglomerative clustering on PCA-reduced data (50 components) for memory efficiency. The dendrogram is truncated to the top levels to remain legible.

In [ ]:
DENDRO_SAMPLE = min(5_000, len(X_clust))
rng_dendro = np.random.RandomState(42)
dendro_idx = rng_dendro.choice(len(X_clust), size=DENDRO_SAMPLE, replace=False)

X_hier = X_clust.iloc[dendro_idx].values

Z = linkage(X_hier, method='ward')

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(
    Z,
    truncate_mode='lastp',
    p=30,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax,
)
ax.set_title(f'Ward Dendrogram (n={DENDRO_SAMPLE:,}, PCA-50, truncated)')
ax.set_xlabel('Cluster Size')
ax.set_ylabel('Distance')
fig.tight_layout()
save_figure(fig, '05_dendrogram.png')
plt.show()

In [ ]:
agg = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage='ward')
agg_labels_sample = agg.fit_predict(X_hier)

km_labels_sample = cluster_labels[dendro_idx]
ari = adjusted_rand_score(km_labels_sample, agg_labels_sample)
print(f'Adjusted Rand Index (K-Means vs Agglomerative, k={OPTIMAL_K}): {ari:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pca2_sub = X_pca2[dendro_idx]
for ax, labels, title in zip(
    axes,
    [km_labels_sample, agg_labels_sample],
    ['K-Means', 'Agglomerative (Ward)'],
):
    scatter = ax.scatter(pca2_sub[:, 0], pca2_sub[:, 1], c=labels, cmap='tab10', alpha=0.4, s=8)
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(*scatter.legend_elements(), title='Cluster', loc='upper right')
fig.suptitle(f'Cluster Comparison on PCA 2-D (ARI = {ari:.3f})', fontsize=13)
fig.tight_layout()
save_figure(fig, '05_kmeans_vs_agglomerative.png')
plt.show()

## 6. Gaussian Mixture Model (Soft Clustering)

GMM allows soft (probabilistic) cluster assignments and provides BIC/AIC model-selection criteria to validate the number of components.

In [ ]:
X_gmm = X_clust.values

bics, aics = [], []
for k in K_RANGE:
    gmm = GaussianMixture(n_components=k, covariance_type='full', random_state=42, max_iter=200)
    gmm.fit(X_gmm)
    bics.append((k, gmm.bic(X_gmm)))
    aics.append((k, gmm.aic(X_gmm)))
    print(f'  k={k}  BIC={bics[-1][1]:.0f}  AIC={aics[-1][1]:.0f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([b[0] for b in bics], [b[1] for b in bics], marker='o', label='BIC')
ax.plot([a[0] for a in aics], [a[1] for a in aics], marker='s', label='AIC')
ax.set_xlabel('Number of Components (k)')
ax.set_ylabel('Information Criterion')
ax.set_title('GMM – BIC / AIC vs. k (PCA-50)')
ax.legend()
fig.tight_layout()
save_figure(fig, '05_gmm_bic_aic.png')
plt.show()

In [ ]:
gmm_final = GaussianMixture(n_components=OPTIMAL_K, covariance_type='full', random_state=42, max_iter=200)
gmm_final.fit(X_gmm)
gmm_labels = gmm_final.predict(X_gmm)
gmm_probs = gmm_final.predict_proba(X_gmm)
gmm_confidence = gmm_probs.max(axis=1)

print(f'GMM (k={OPTIMAL_K}) fitted.')
print(f'Mean membership confidence: {gmm_confidence.mean():.3f}')
print(f'Cluster sizes: {np.bincount(gmm_labels)}')

cmap = plt.cm.tab10
norm_labels = gmm_labels / max(1, gmm_labels.max()) if gmm_labels.max() > 0 else gmm_labels.astype(float)
rgba = cmap(norm_labels)
rgba[:, 3] = np.clip(gmm_confidence * 0.7 + 0.1, 0, 1)

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=rgba, s=6)

handles = []
for cid in range(OPTIMAL_K):
    color = cmap(cid / max(1, OPTIMAL_K - 1)) if OPTIMAL_K > 1 else cmap(0.0)
    handles.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=8, label=f'{cid}'))
ax.legend(handles=handles, title='Component', loc='upper right')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'GMM Soft Clustering (k={OPTIMAL_K}), alpha = membership confidence')
fig.tight_layout()
save_figure(fig, '05_gmm_scatter.png')
plt.show()

ari_gmm_km = adjusted_rand_score(cluster_labels, gmm_labels)
print(f'ARI (K-Means vs GMM): {ari_gmm_km:.4f}')

## 7. Association Rule Mining (Readmission-Focused)

Notebook 01 performed broad association rule mining on the raw data. Here we work with the processed (one-hot encoded) features and focus specifically on rules whose consequent involves early readmission. This extends NB 01 by mining on the cleaned, deduplicated dataset used for modelling.

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

MEDICATION_PREFIXES = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
    'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
    'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone',
]
DEMOGRAPHIC_PREFIXES = ['age', 'race', 'gender']
CLINICAL_PREFIXES = ['max_glu_serum', 'A1Cresult', 'change', 'diabetesMed']

KEEP_PREFIXES = MEDICATION_PREFIXES + DEMOGRAPHIC_PREFIXES + CLINICAL_PREFIXES

ar_cols = [c for c in X.columns if any(c == p or c.startswith(p + '_') for p in KEEP_PREFIXES)]

ar_df = X[ar_cols].copy()
ar_df['readmitted_binary'] = y.values

ar_bool = ar_df.astype(bool)

widely_prescribed = {'metformin', 'insulin', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone'}
drop_no_cols = [
    c for c in ar_bool.columns
    if '_No' in c
    and any(c.startswith(p) for p in MEDICATION_PREFIXES)
    and not any(c.startswith(w) for w in widely_prescribed)
]
ar_bool = ar_bool.drop(columns=drop_no_cols, errors='ignore')

print(f'Transaction matrix: {ar_bool.shape[0]:,} rows × {ar_bool.shape[1]} items')
ar_bool.head()

In [ ]:
frequent_itemsets = fpgrowth(ar_bool, min_support=0.05, use_colnames=True)
print(f'Frequent itemsets found: {len(frequent_itemsets):,}')

rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)
print(f'Association rules (lift >= 1.2): {len(rules):,}')

readmit_consequent = rules[
    rules['consequents'].apply(lambda s: 'readmitted_binary' in str(s))
].copy()
readmit_consequent = readmit_consequent.sort_values('confidence', ascending=False).reset_index(drop=True)
print(f'Rules with readmitted_binary in consequent: {len(readmit_consequent)}')

display(readmit_consequent[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20))

In [ ]:
if len(readmit_consequent) > 0:
    fig = plot_association_rules(
        readmit_consequent,
        title='Association Rules — Readmission in Consequent (support vs confidence, colour = lift)',
    )
    save_figure(fig, '05_association_rules_readmission.png')
    plt.show()

    export_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage', 'conviction']
    export_cols = [c for c in export_cols if c in readmit_consequent.columns]
    save_df = readmit_consequent[export_cols].head(50).copy()
    save_df['antecedents'] = save_df['antecedents'].apply(lambda s: ', '.join(sorted(s)))
    save_df['consequents'] = save_df['consequents'].apply(lambda s: ', '.join(sorted(s)))
    save_df.to_csv(TABLES_DIR / '05_top_readmission_rules.csv', index=False)
    print(f'Saved to {TABLES_DIR / "05_top_readmission_rules.csv"}')
else:
    print('No rules with readmitted_binary in consequent found at these thresholds.')

## 8. Cluster-Stratified Readmission Profiles

This section bridges unsupervised structure with the supervised target by profiling each K-Means cluster in terms of clinical features and readmission patterns.

In [ ]:
profile_extended = df.groupby('cluster').agg(
    n=('cluster', 'size'),
    readmit_rate=(TARGET, 'mean'),
    **{f'{c}_mean': (c, 'mean') for c in PROFILE_COLS},
).round(3)

display(profile_extended)

heatmap_data = df.groupby('cluster')[PROFILE_COLS].mean()
heatmap_z = (heatmap_data - heatmap_data.mean()) / heatmap_data.std()

fig, ax = plt.subplots(figsize=(10, max(3, OPTIMAL_K * 0.8 + 1)))
sns.heatmap(
    heatmap_z,
    annot=heatmap_data.round(1).values,
    fmt='',
    cmap='RdYlBu_r',
    center=0,
    linewidths=0.5,
    ax=ax,
    yticklabels=[f'Cluster {i}' for i in range(OPTIMAL_K)],
)
ax.set_title('Cluster Feature Profiles (z-scored colour, raw values annotated)')
fig.tight_layout()
save_figure(fig, '05_cluster_feature_heatmap.png')
plt.show()

### 8.1 Per-Cluster Association Rules

Re-run FP-Growth within each cluster to discover whether different patient subgroups have distinct readmission-associated patterns.

In [ ]:
cluster_rules_summary = {}
for cid in range(OPTIMAL_K):
    mask = df['cluster'] == cid
    ar_sub = ar_bool.loc[mask]
    n_sub = len(ar_sub)

    try:
        fi_sub = fpgrowth(ar_sub, min_support=0.05, use_colnames=True)
        if len(fi_sub) == 0:
            print(f'Cluster {cid} (n={n_sub:,}): no frequent itemsets at min_support=0.05')
            continue
        rules_sub = association_rules(fi_sub, metric='lift', min_threshold=1.2)
        readmit_sub = rules_sub[
            rules_sub['consequents'].apply(lambda s: 'readmitted_binary' in str(s))
        ].sort_values('confidence', ascending=False)

        cluster_rules_summary[cid] = readmit_sub
        print(f'Cluster {cid} (n={n_sub:,}): {len(readmit_sub)} readmission-consequent rules')
        if len(readmit_sub) > 0:
            display(readmit_sub[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5))
    except Exception as e:
        print(f'Cluster {cid} (n={n_sub:,}): rule mining failed — {e}')

## 9. Summary and Interpretation

### Dimensionality Reduction
- **PCA** reveals how many latent dimensions are needed to capture the variance in the ~2,400-feature one-hot space. The cumulative variance curve guides component selection for downstream methods (t-SNE, hierarchical clustering, GMM all use PCA-50 pre-reduction).
- The **PCA 2-D projection** and **t-SNE / UMAP embeddings** visualise whether readmitted vs non-readmitted patients occupy separable regions. Heavy overlap would confirm that readmission risk is not driven by a small number of dominant clinical axes — consistent with the modest supervised F1 scores observed in Notebook 3.

### Clustering
- **K-Means** discovers patient phenotype clusters. The elbow and silhouette analyses identify a data-driven *k* rather than the placeholder k=3 from Notebook 3.
- **Agglomerative clustering** (Ward linkage) provides a complementary partitioning; the Adjusted Rand Index quantifies agreement with K-Means. High ARI indicates the cluster structure is robust to algorithmic choice.
- **Gaussian Mixture Models** add soft cluster assignments and BIC/AIC-based model selection, allowing assessment of cluster uncertainty per patient.

### Cluster–Readmission Bridge
- The **readmission rate bar chart** and **chi-squared test** establish whether cluster membership is statistically associated with 30-day readmission — a prerequisite for using cluster identity as a feature or risk-stratification tool.
- The **feature heatmap** profiles each cluster by clinical utilisation metrics (lab procedures, medications, inpatient stays, diagnoses), enabling clinical naming of clusters (e.g. "high-complexity multi-morbid" vs "low-acuity single-encounter").

### Association Rules
- Readmission-consequent rules surface **interpretable clinical co-occurrence patterns** linked to early readmission, extending the broad exploratory rules from Notebook 1.
- Per-cluster rule mining reveals whether readmission drivers differ across patient phenotypes — actionable for **targeted intervention design** (e.g. medication counselling in one cluster, discharge planning in another).

### Integration with Supervised Modelling (Notebook 3)
- Cluster membership could serve as an additional feature for supervised classifiers.
- Association rules provide human-readable decision aids that complement the black-box LightGBM model's predictions.
- Together, unsupervised structure and supervised prediction support the project's dual goal of **accurate risk scoring** and **interpretable clinical insight** under the HRRP cost framework.